In [2]:
import pandas as pd
import re
import numpy as np

# ==============================
# DOSYA AYARLARI
# ==============================

GIRIS_DOSYASI = "temiz_laptop_yorumlari.xlsx"
CIKIS_DOSYASI = "yorum_ozellikleri.xlsx"

df = pd.read_excel(GIRIS_DOSYASI)

print("Temiz yorum dosyası okundu.")
print("Toplam yorum:", len(df))


# ==============================
# KELİME GRUPLARI
# ==============================

olumlu_kelimeler = [
    "iyi", "güzel", "hızlı", "kaliteli", "memnun",
    "mükemmel", "harika", "başarılı", "yeterli",
    "tavsiye", "sessiz", "beğendim", "sorunsuz",
    "sağlam", "şık", "akıcı", "güçlü"
]

olumsuz_kelimeler = [
    "kötü", "ısınma", "ısınıyor", "fan", "gürültü",
    "ses", "donma", "kasma", "servis", "batarya",
    "şarj", "problem", "sorun", "bozuk", "yavaş",
    "iade", "eksik", "hasarlı", "çizik"
]

fp_kelimeler = [
    "fiyat performans", "f/p", "fp", "performans", "fiyat"
]

oyun_kelimeler = [
    "oyun", "valorant", "cs2", "gta", "pubg", "lol",
    "fps", "gaming", "oyuncu"
]

profesyonel_kelimeler = [
    "autocad", "solidworks", "python", "yazılım", "programlama",
    "render", "tasarım", "kodlama", "mühendislik"
]

isinma_kelimeler = [
    "ısınma", "ısınıyor", "ısındı", "sıcak"
]

fan_kelimeler = [
    "fan", "ses", "gürültü", "sesli"
]

batarya_kelimeler = [
    "batarya", "pil", "şarj"
]


# ==============================
# YARDIMCI FONKSİYONLAR
# ==============================

def metin_temizle(metin):
    if pd.isna(metin):
        return ""

    metin = str(metin).lower()
    metin = re.sub(r"\s+", " ", metin)
    return metin.strip()


def kelime_say(metin, kelime_listesi):
    toplam = 0

    for kelime in kelime_listesi:
        kelime = kelime.lower()

        if " " in kelime:
            toplam += metin.count(kelime)
        else:
            toplam += len(re.findall(rf"\b{re.escape(kelime)}\b", metin))

    return toplam


def toplam_kelime_sayisi(metin):
    kelimeler = re.findall(r"\b\w+\b", metin)
    return len(kelimeler)


def guvenli_bolme(pay, payda):
    if payda == 0:
        return 0
    return pay / payda


def risk_seviyesi(risk_orani):
    if risk_orani < 0.01:
        return "Düşük"
    elif risk_orani < 0.03:
        return "Orta"
    else:
        return "Yüksek"


def memnuniyet_seviyesi(skor):
    if skor >= 0.03:
        return "Yüksek"
    elif skor >= 0.01:
        return "Orta"
    else:
        return "Düşük"


# ==============================
# YORUM METİNLERİNİ HAZIRLA
# ==============================

df["yorum_temiz"] = df["yorum_metni"].apply(metin_temizle)
df["yorum_kelime_sayisi"] = df["yorum_temiz"].apply(toplam_kelime_sayisi)


# ==============================
# HER YORUM İÇİN KELİME SAYILARI
# ==============================

df["olumlu_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, olumlu_kelimeler)
)

df["olumsuz_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, olumsuz_kelimeler)
)

df["fp_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, fp_kelimeler)
)

df["oyun_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, oyun_kelimeler)
)

df["profesyonel_kullanim_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, profesyonel_kelimeler)
)

df["isinma_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, isinma_kelimeler)
)

df["fan_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, fan_kelimeler)
)

df["batarya_kelime_sayisi"] = df["yorum_temiz"].apply(
    lambda x: kelime_say(x, batarya_kelimeler)
)


# ==============================
# ÜRÜN BAZLI ÖZETLEME
# ==============================

urun_ozet = df.groupby(["ürün_id", "ürün_adı"]).agg(
    cekilen_yorum_sayisi=("yorum_metni", "count"),
    toplam_yorum_kelime_sayisi=("yorum_kelime_sayisi", "sum"),
    ortalama_yorum_uzunlugu=("yorum_kelime_sayisi", "mean"),

    olumlu_kelime_sayisi=("olumlu_kelime_sayisi", "sum"),
    olumsuz_kelime_sayisi=("olumsuz_kelime_sayisi", "sum"),
    fp_kelime_sayisi=("fp_kelime_sayisi", "sum"),
    oyun_kelime_sayisi=("oyun_kelime_sayisi", "sum"),
    profesyonel_kullanim_kelime_sayisi=("profesyonel_kullanim_kelime_sayisi", "sum"),
    isinma_kelime_sayisi=("isinma_kelime_sayisi", "sum"),
    fan_kelime_sayisi=("fan_kelime_sayisi", "sum"),
    batarya_kelime_sayisi=("batarya_kelime_sayisi", "sum")
).reset_index()


# ==============================
# GELİŞMİŞ ORAN VE SKORLAR
# ==============================

urun_ozet["olumlu_oran"] = urun_ozet.apply(
    lambda row: guvenli_bolme(
        row["olumlu_kelime_sayisi"],
        row["toplam_yorum_kelime_sayisi"]
    ),
    axis=1
)

urun_ozet["olumsuz_oran"] = urun_ozet.apply(
    lambda row: guvenli_bolme(
        row["olumsuz_kelime_sayisi"],
        row["toplam_yorum_kelime_sayisi"]
    ),
    axis=1
)

urun_ozet["risk_kelime_sayisi"] = (
    urun_ozet["isinma_kelime_sayisi"] +
    urun_ozet["fan_kelime_sayisi"] +
    urun_ozet["batarya_kelime_sayisi"]
)

urun_ozet["risk_orani"] = urun_ozet.apply(
    lambda row: guvenli_bolme(
        row["risk_kelime_sayisi"],
        row["toplam_yorum_kelime_sayisi"]
    ),
    axis=1
)

urun_ozet["yorum_memnuniyet_skoru"] = urun_ozet.apply(
    lambda row: guvenli_bolme(
        row["olumlu_kelime_sayisi"] - row["olumsuz_kelime_sayisi"],
        row["toplam_yorum_kelime_sayisi"]
    ),
    axis=1
)

urun_ozet["normalize_memnuniyet_skoru"] = urun_ozet.apply(
    lambda row: guvenli_bolme(
        row["olumlu_kelime_sayisi"],
        row["olumlu_kelime_sayisi"] + row["olumsuz_kelime_sayisi"] + 1
    ),
    axis=1
)

urun_ozet["risk_seviyesi"] = urun_ozet["risk_orani"].apply(risk_seviyesi)

urun_ozet["memnuniyet_seviyesi"] = urun_ozet["yorum_memnuniyet_skoru"].apply(
    memnuniyet_seviyesi
)


# ==============================
# KAYDET
# ==============================

urun_ozet.to_excel(CIKIS_DOSYASI, index=False)

print("\nYorum özellikleri dosyası oluşturuldu.")
print("Dosya:", CIKIS_DOSYASI)
print("Ürün sayısı:", len(urun_ozet))

print("\nÖrnek çıktı:")
print(urun_ozet.head())

Temiz yorum dosyası okundu.
Toplam yorum: 2914

Yorum özellikleri dosyası oluşturuldu.
Dosya: yorum_ozellikleri.xlsx
Ürün sayısı: 269

Örnek çıktı:
   ürün_id                                           ürün_adı  \
0        1  Lenovo Ideapad Slim 3 AMD Ryzen 7 7735HS 16GB ...   
1        2  Lenovo Ideapad Slim 3 AMD Ryzen 5 7535HS 16GB ...   
2        3  Medion Signium 14 S1 MD600032 Intel Core 5 120...   
3        4  HP AI 15 Intel Core Ultra 5 225U 16GB 512GB SS...   
4        5  Huawei MateBook D 16 2024 16" i5-12450H UMA 16...   

   cekilen_yorum_sayisi  toplam_yorum_kelime_sayisi  ortalama_yorum_uzunlugu  \
0                    25                         796                31.840000   
1                     4                          51                12.750000   
2                    14                         531                37.928571   
3                    18                         564                31.333333   
4                     8                         252          